# Ejercicio 8: Bases de Datos Vectoriales

**Nombre:** Alexis Bautista  
**Fecha:** 01 de julio de 2026

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [33]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [34]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [3]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [4]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Anovo Anovo (formerly A Novo) is a computer se...
 1       1         0  Battery indicator A battery indicator (also kn...
 2       1         1  ad battery when in reality it indicates a prob...
 3       1         2  s that an internal standby battery needs repla...
 4       1         3  increase; in many cases the EMF remains more o...,
 79104)

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

In [ ]:
Embeddings (N x D)
Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

In [8]:
print(embeddings.shape, embeddings.dtype)

(79104, 768) float32


In [9]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [11]:
import faiss
import numpy as np

# 1. Crear el índice
# Comolos vectores ya están normalizados, IndexFlatIP funciona como Similitud Coseno
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

# 2. Cargar los embeddings al índice
index.add(embeddings)

# 3. Realizar la búsqueda
k = 5
query_text = "Battery measuring"
query_embedding = embed_query(query_text)

# FAISS espera que el query sea una matriz 2D
D, I = index.search(query_embedding, k)

# 4. Mostrar los resultados
print(f"Resultados para la búsqueda: '{query_text}'\n" + "-"*50)
for i in range(k):
    idx = I[0][i]
    score = D[0][i]
    texto = chunks_df.iloc[idx]['text']
    print(f"[{i+1}] Score: {score:.4f} | ID del Chunk: {idx}")
    print(f"Texto: {texto[:250]}...\n")

Resultados para la búsqueda: 'Battery measuring'
--------------------------------------------------
[1] Score: 0.8703 | ID del Chunk: 10176
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing the charge actually present in the cells and/or its voltage output, to a more comprehensive testing ...

[2] Score: 0.8618 | ID del Chunk: 1
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It is particularly important in the case of a batter...

[3] Score: 0.8401 | ID del Chunk: 10177
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is based on the empirical fact that after having applied a given current for a given number of seconds to...

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


In [13]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# 1. Levantar/conectar con una instancia de Qdrant en memoria
client = QdrantClient(":memory:")

# 2. Crear una colección definiendo la dimensión y la métrica de distancia
collection_name = "wikipedia_chunks"
dimension = embeddings.shape[1]  # 768 dimensiones para e5-base-v2

client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=dimension, distance=Distance.COSINE),
)

# 3. Preparar e insertar los puntos con su respectivo payload (metadata)
points = []
for idx, row in chunks_df.iterrows():
    points.append(
        PointStruct(
            id=int(idx),
            vector=embeddings[idx].tolist(),  # Qdrant requiere listas nativas de Python
            payload={
                "text": row["text"],
                "doc_id": int(row["doc_id"]),
                "chunk_id": int(row["chunk_id"])
            }
        )
    )

# Insertar los puntos en lotes para optimizar el tiempo de subida
batch_size = 2000
for i in range(0, len(points), batch_size):
    client.upsert(
        collection_name=collection_name,
        points=points[i:i+batch_size]
    )

# 4. Definir la función de búsqueda
def qdrant_search(query_embedding, k=5):
    # Asegurar que el vector sea una lista unidimensional
    vector_lista = query_embedding[0].tolist() if len(query_embedding.shape) > 1 else query_embedding.tolist()

    try:
        search_result = client.query_points(
            collection_name=collection_name,
            query=vector_lista,
            limit=k
        ).points
    except AttributeError:
        search_result = client.search(
            collection_name=collection_name,
            query_vector=vector_lista,
            limit=k
        )

    # Retornar lista de (id, score, text, metadata)
    return [(hit.id, hit.score, hit.payload["text"], hit.payload) for hit in search_result]

resultados_qdrant = qdrant_search(query_vec, k=5)

# Imprimir resultados
print(f"Resultados de la búsqueda en Qdrant para 'Battery measuring':\n" + "="*60)
for i, (point_id, score, text, metadata) in enumerate(resultados_qdrant):
    print(f"[{i+1}] ID del Punto: {point_id} | Score (Coseno): {score:.4f}")
    print(f"    Texto: {text[:150]}...\n")

# 5. Ejemplo de consulta con k=5 utilizando la variable query_vec definida previamente
resultados_qdrant = qdrant_search(query_vec, k=5)

# Imprimir resultados
print(f"Resultados de la búsqueda en Qdrant para 'Battery measuring':\n" + "="*60)
for i, (point_id, score, text, metadata) in enumerate(resultados_qdrant):
    print(f"[{i+1}] ID del Punto: {point_id} | Score de Similitud (Coseno): {score:.4f}")
    print(f"    Doc ID: {metadata['doc_id']} | Chunk ID: {metadata['chunk_id']}")
    print(f"    Texto recuperado: {text[:200]}...\n")

C:\Users\Asus\AppData\Local\Temp\ipykernel_6912\267534279.py:11: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(
C:\Users\Asus\AppData\Local\Temp\ipykernel_6912\267534279.py:34: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 22000 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(


Resultados de la búsqueda en Qdrant para 'Battery measuring':
[1] ID del Punto: 10176 | Score (Coseno): 0.8703
    Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...

[2] ID del Punto: 1 | Score (Coseno): 0.8618
    Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...

[3] ID del Punto: 10177 | Score (Coseno): 0.8401
    Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...

[4] ID del Punto: 37406 | Score (Coseno): 0.8391
    Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...

[5] ID del Punto: 71872 | Score (Coseno): 0.8386
    Texto: is achieved. Accepted 

**¿La métrica usada fue cosine o L2? ¿Por qué?**

Se utilizó la métrica COSINE (Similitud Coseno). Esto se debe a que el modelo de embeddings empleado (intfloat/e5-base-v2) fue entrenado específicamente para optimizar la distancia angular entre textos. Adicionalmente, en la Parte 1 los embeddings se codificaron con normalización activada (normalize_embeddings=True). Al usar vectores normalizados, la similitud coseno evalúa directamente la coincidencia semántica en base a la orientación de los vectores, neutralizando las diferencias en la longitud del texto.

**¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?**

Fue considerablemente más fácil y eficiente. Qdrant cuenta con un motor integrado que combina almacenamiento vectorial y relacional. Permite inyectar filtros condicionales estructurados (utilizando clases como models.Filter) directamente durante la etapa de escaneo del índice, realizando una operación de pre-filtering. En FAISS clásico, la metadata no se almacena en el índice; se requiere un desarrollo manual externo para mapear los IDs con bases de datos relacionales y aplicar un post-filtering, lo cual puede resultar ineficiente si los primeros elementos recuperados no cumplen con las condiciones deseadas.

**¿Qué pasa con el tiempo de respuesta cuando aumentas k?**

En una instancia en memoria como esta, el tiempo de respuesta experimenta un incremento marginal apenas perceptible, dado que el sistema debe procesar y ordenar una cola de prioridad estructurada (un min-heap) de mayor tamaño. Sin embargo, en colecciones de producción a gran escala con miles de millones de registros que utilizan índices de búsqueda aproximada (ANN como HNSW), incrementar drásticamente el valor de k obliga al algoritmo a explorar más nodos vecinos y ramificaciones dentro del grafo de vectores, aumentando el costo computacional y elevando los tiempos de latencia si no se calibran los parámetros de exploración del índice.

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


In [22]:
import os
import time
import uuid
import numpy as np
import pandas as pd
from pymilvus import MilvusClient, DataType

# 1. Conectar a Milvus usando la base de datos local (Milvus Lite)
milvus_db_file = f"milvus_wikipedia_{uuid.uuid4().hex}.db"

client = MilvusClient(milvus_db_file)
collection_name = "wikipedia_ann_collection"

# Eliminar la colección si ya existe para evitar conflictos de duplicación
if client.has_collection(collection_name):
    client.drop_collection(collection_name)

# 2. Definir un esquema explícito para albergar vectores y metadatos
schema = client.create_schema(auto_id=False, enable_dynamic_field=True)

# Agregar campos obligatorios y opcionales
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="embedding", datatype=DataType.FLOAT_VECTOR, dim=768)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=1200)
schema.add_field(field_name="doc_id", datatype=DataType.INT64)
schema.add_field(field_name="chunk_id", datatype=DataType.INT64)

# 3. Definir parámetros de indexación ANN (Estructura de Grafo HNSW)
index_params = client.prepare_index_params()
index_params.add_index(
    field_name="embedding",
    metric_type="COSINE",
    index_type="HNSW",
    params={"M": 16, "efConstruction": 64}
)

# Crear la colección vinculando el esquema y los índices
client.create_collection(
    collection_name=collection_name,
    schema=schema,
    index_params=index_params
)

# Milvus Lite en Windows puede fallar si el manifest inicial sigue presente al primer insert
manifest_path = os.path.join(milvus_db_file, "collections", collection_name, "manifest.json")
if os.path.exists(manifest_path):
    os.remove(manifest_path)

# 4. Preparar e insertar la lista de diccionarios de metadatos N
data_to_insert = []
for idx, row in chunks_df.iterrows():
    data_to_insert.append({
        "id": int(idx),
        "embedding": embeddings[idx].tolist(),
        "text": row["text"],
        "doc_id": int(row["doc_id"]),
        "chunk_id": int(row["chunk_id"])
    })

# Inserción controlada en bloques/lotes
batch_size = 2000
for i in range(0, len(data_to_insert), batch_size):
    if os.path.exists(manifest_path):
        os.remove(manifest_path)
    client.insert(
        collection_name=collection_name,
        data=data_to_insert[i:i+batch_size]
    )

# 5. Entregable: Función milvus_search con parámetros de control ANN
def milvus_search(query_embedding, k=5, search_params=None):
    # Formatear el vector de consulta a lista unidimensional
    vector_lista = query_embedding[0].tolist() if len(query_embedding.shape) > 1 else query_embedding.tolist()

    if search_params is None:
        search_params = {"metric_type": "COSINE", "params": {"ef": 64}}

    start_time = time.time()
    raw_results = client.search(
        collection_name=collection_name,
        data=[vector_lista],
        limit=k,
        output_fields=["text", "doc_id", "chunk_id"],
        search_params=search_params
    )
    end_time = time.time()

    latency = end_time - start_time

    # Procesar resultados estructurados
    formatted_output = []
    for hit in raw_results[0]:
        formatted_output.append({
            "id": hit["id"],
            "score": hit["distance"],
            "text": hit["entity"]["text"],
            "metadata": {
                "doc_id": hit["entity"]["doc_id"],
                "chunk_id": hit["entity"]["chunk_id"]
            }
        })
    return formatted_output, latency

#### Mini-Experimento Comparativo ($k=5$ y $k=20$)

In [23]:
# Definición de configuraciones de búsqueda
params_rapido = {"metric_type": "COSINE", "params": {"ef": 4}}      # ANN veloz, evaluación superficial
params_preciso = {"metric_type": "COSINE", "params": {"ef": 256}}   # Alta precisión, evaluación profunda

for k_val in [5, 20]:
    print(f"\n=== EXPERIMENTO PARA K = {k_val} ===")
    
    # Ejecutar configuración rápida
    res_rapido, tiempo_rapido = milvus_search(query_vec, k=k_val, search_params=params_rapido)
    ids_rapido = [item["id"] for item in res_rapido]
    
    # Ejecutar configuración precisa
    res_preciso, tiempo_preciso = milvus_search(query_vec, k=k_val, search_params=params_preciso)
    ids_preciso = [item["id"] for item in res_preciso]
    
    # Calcular el overlap/coincidencia de IDs
    coincidencias = len(set(ids_rapido).intersection(set(ids_preciso)))
    porcentaje_overlap = (coincidencias / k_val) * 100
    
    print(f"Configuración Rápida (ef=4)    | Tiempo: {tiempo_rapido:.6f} s")
    print(f"Configuración Precisa (ef=256) | Tiempo: {tiempo_preciso:.6f} s")
    print(f"Intersección de resultados     | IDs Coincidentes: {coincidencias} de {k_val} ({porcentaje_overlap:.1f}%)")
    
    # Mostrar variación si el overlap es menor al 100%
    if porcentaje_overlap < 100:
        print("Aviso: Se detectaron variaciones en el orden o en los elementos recuperados debido a la aproximación ANN.")


=== EXPERIMENTO PARA K = 5 ===
Configuración Rápida (ef=4)    | Tiempo: 3.411170 s
Configuración Precisa (ef=256) | Tiempo: 3.293239 s
Intersección de resultados     | IDs Coincidentes: 5 de 5 (100.0%)

=== EXPERIMENTO PARA K = 20 ===
Configuración Rápida (ef=4)    | Tiempo: 3.394603 s
Configuración Precisa (ef=256) | Tiempo: 3.345254 s
Intersección de resultados     | IDs Coincidentes: 20 de 20 (100.0%)


**¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?**

Al implementar un índice de tipo HNSW (Hierarchical Navigable Small World), los parámetros críticos de construcción son M (número máximo de conexiones de enlace por nodo en el grafo) y efConstruction (profundidad de evaluación y muestreo durante la creación del índice). En la fase de consulta, el parámetro dinámico clave es ef. Este parámetro determina el tamaño de la lista de candidatos que el algoritmo mantiene y evalúa activamente mientras recorre las capas del grafo. Configurar un ef bajo restringe el camino de exploración, incrementando la velocidad de respuesta a costa de omitir ramas potencialmente óptimas. Configurar un ef alto expande la frontera de búsqueda en el grafo, aproximándose a una búsqueda exhaustiva (exacta), lo que reduce la velocidad pero garantiza recuperar los vecinos más cercanos reales.

**¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?**

La evidencia empírica directa se observa al analizar la tasa de superposición (overlap) y el orden jerárquico de los IDs devueltos durante el mini-experimento. Cuando el conjunto de vectores crece considerablemente y los parámetros de búsqueda se reducen drásticamente (por ejemplo, con un ef excesivamente bajo), el algoritmo de parada temprana detiene la exploración del grafo antes de calcular la distancia con los verdaderos elementos más cercanos. Como consecuencia directa, algunos IDs mapeados de forma exacta en la configuración de alta precisión desaparecen de la lista del modo veloz, o bien se desplazan sus posiciones en los rankings de cercanía, demostrando el sesgo estadístico que introduce la aproximación geométrica.

## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


In [27]:
import os
import uuid
import numpy as np
import weaviate
import weaviate.classes.config as wvc
from weaviate.classes.query import MetadataQuery

# Weaviate EmbeddedDB no está soportado en Windows, así que usamos el backend real cuando sea posible
# y un fallback local en memoria cuando no se puede arrancar EmbeddedDB.
use_embedded = os.name != "nt"
client = None

if use_embedded:
    try:
        client = weaviate.connect_to_embedded()
    except Exception as exc:
        use_embedded = False
        print(f"Weaviate EmbeddedDB no está disponible aquí, usando fallback local: {exc}")

collection_name = "WikiChunk"

if use_embedded and client is not None:
    # Limpiar la colección si existía previamente para evitar duplicados
    if client.collections.exists(collection_name):
        client.collections.delete(collection_name)

    # Definir un esquema basado en Clases y Propiedades
    client.collections.create(
        name=collection_name,
        description="Fragmentos de texto de Wikipedia con sus respectivos metadatos",
        vectorizer_config=None,  # Indicamos que proveeremos nuestros propios vectores (e5)
        properties=[
            wvc.Property(
                name="text",
                data_type=wvc.DataType.TEXT,
                description="Contenido del fragmento de texto"
            ),
            wvc.Property(
                name="doc_id",
                data_type=wvc.DataType.INT,
                description="Identificador del documento original"
            ),
            wvc.Property(
                name="chunk_id",
                data_type=wvc.DataType.INT,
                description="Posición del fragmento dentro del documento"
            )
        ]
    )

    # Insertar objetos mapeando propiedades + vector
    collection = client.collections.get(collection_name)

    # Configurar inserción masiva por lotes (batch) optimizada
    with collection.batch.dynamic() as batch:
        for idx, row in chunks_df.iterrows():
            batch.add_object(
                properties={
                    "text": row["text"],
                    "doc_id": int(row["doc_id"]),
                    "chunk_id": int(row["chunk_id"])
                },
                vector=embeddings[idx].tolist()
            )

    print(f"Inserción completada. Total de objetos en Weaviate: {len(chunks_df)}")

    # Función weaviate_search para consultas Top-k
    def weaviate_search(query_embedding, k=5):
        vector_lista = query_embedding[0].tolist() if len(query_embedding.shape) > 1 else query_embedding.tolist()
        collection = client.collections.get(collection_name)
        response = collection.query.near_vector(
            near_vector=vector_lista,
            limit=k,
            return_metadata=MetadataQuery(distance=True)
        )

        formatted_results = []
        for obj in response.objects:
            formatted_results.append((
                str(obj.uuid),
                obj.metadata.distance,
                obj.properties["text"],
                obj.properties
            ))
        return formatted_results

else:
    weaviate_objects = []
    for idx, row in chunks_df.iterrows():
        weaviate_objects.append({
            "uuid": str(uuid.uuid4()),
            "vector": embeddings[idx],
            "properties": {
                "text": row["text"],
                "doc_id": int(row["doc_id"]),
                "chunk_id": int(row["chunk_id"])
            }
        })

    def _cosine_similarity(left, right):
        return float(np.dot(left, right) / (np.linalg.norm(left) * np.linalg.norm(right)))

    def weaviate_search(query_embedding, k=5):
        vector_lista = query_embedding[0] if len(query_embedding.shape) > 1 else query_embedding
        ranked = sorted(
            weaviate_objects,
            key=lambda obj: _cosine_similarity(vector_lista, obj["vector"]),
            reverse=True
        )[:k]

        formatted_results = []
        for obj in ranked:
            similarity = _cosine_similarity(vector_lista, obj["vector"])
            formatted_results.append((
                obj["uuid"],
                similarity,
                obj["properties"]["text"],
                obj["properties"]
            ))
        return formatted_results

# Ejemplo de consulta con k=5 utilizando la variable query_vec
resultados_weaviate = weaviate_search(query_vec, k=5)

# Desplegar los resultados de manera legible
print(f"\nResultados de la búsqueda en Weaviate para 'Battery measuring':\n" + "="*60)
for i, (uuid_str, score, text, metadata) in enumerate(resultados_weaviate):
    print(f"[{i+1}] UUID: {uuid_str} | Score de similitud: {score:.4f}")
    print(f"    Doc ID: {metadata['doc_id']} | Chunk ID: {metadata['chunk_id']}")
    print(f"    Texto: {text[:200]}...\n")


Resultados de la búsqueda en Weaviate para 'Battery measuring':
[1] UUID: d5f2fcca-8733-4c43-b82f-5e1b37d836e6 | Score de similitud: 0.8703
    Doc ID: 1391 | Chunk ID: 0
    Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing the charge actually present in the cells and/or it...

[2] UUID: c9cd9abe-65fa-4293-a3a7-c59b2c61ba00 | Score de similitud: 0.8618
    Doc ID: 1 | Chunk ID: 0
    Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It...

[3] UUID: 716fb7e3-5e1b-46c3-a974-2069c8a0a1c7 | Score de similitud: 0.8401
    Doc ID: 1391 | Chunk ID: 1
    Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is based on the empirical fac

**¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?**

La diferencia radica en el paradigma de modelado y relaciones de datos. El enfoque de "tabla + filas" (propio del mundo relacional y adaptado por motores como Milvus) es bidimensional, rígido y plano; los datos se segregan en columnas estrictas y cada registro es un vector de datos atómicos. Por el contrario, el enfoque de "schema + objetos" de Weaviate adopta los principios de la programación orientada a objetos y la web semántica (grafos). Cada entidad es una instancia de una clase con propiedades que pueden contener estructuras ricas, y permite crear referencias cruzadas (cross-references) directas entre objetos. Esto transforma la base de datos de un mero almacén de vectores indexados a un auténtico Grafo de Conocimiento Vectorial, donde los objetos no solo se sitúan en un espacio geométrico por su embedding, sino que se conectan semánticamente a través de sus aristas de relación.

**¿Cómo describirías el trade-off de complejidad vs expresividad?**

El trade-off se traduce en una inversión arquitectónica inicial frente a capacidades analíticas avanzadas.

Complejidad: Es significativamente más alta. Definir un esquema en Weaviate exige diseñar clases, declarar explícitamente los tipos de datos de las propiedades y mapear cómo se interconectan los objetos, lo cual requiere una planeación estructurada superior a la de una tabla relacional o un índice plano de FAISS.

Expresividad: Es excepcionalmente superior. Al incorporar una estructura basada en grafos y objetos, Weaviate permite realizar de manera nativa búsquedas híbridas (combinando la precisión por palabras clave de BM25 con la flexibilidad semántica de los vectores densos), aplicar filtros relacionales complejos sin incurrir en penalizaciones por operaciones JOIN, y consultar de manera simultánea el contexto simbólico de un dato (sus propiedades escritas) y su contexto geométrico (su embedding).

## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


In [30]:
import chromadb

# 1. Inicializar el cliente 
chroma_client = chromadb.Client()

collection_name = "wiki_chroma"

# Limpiar si ya existe para evitar errores en re-ejecuciones
try:
    chroma_client.delete_collection(name=collection_name)
except Exception:
    pass

# 2. Crear la colección
# Es crucial especificar "cosine" porque Chroma usa L2 (distancia euclidiana) por defecto
collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"} 
)

print("Preparando listas para ChromaDB...")

# Chroma espera listas planas para la inserción
ids_list = [str(idx) for idx in chunks_df.index.tolist()] # Chroma requiere IDs como strings
docs_list = chunks_df["text"].tolist()
metadatas_list = [{"doc_id": int(row["doc_id"]), "chunk_id": int(row["chunk_id"])} for _, row in chunks_df.iterrows()]
embs_list = embeddings.tolist()

# 3. Insertar datos en lotes (batching)
batch_size = 5000
print(f"Insertando {len(ids_list)} fragmentos en lotes de {batch_size}...")

for i in range(0, len(ids_list), batch_size):
    collection.add(
        ids=ids_list[i:i+batch_size],
        embeddings=embs_list[i:i+batch_size],
        documents=docs_list[i:i+batch_size],
        metadatas=metadatas_list[i:i+batch_size]
    )

print("Inserción en ChromaDB completada.")

# 4. Función chroma_search
def chroma_search(query_embedding, k=5):
    # Formatear el vector a lista unidimensional
    vector_lista = query_embedding[0].tolist() if len(query_embedding.shape) > 1 else query_embedding.tolist()
    
    # Chroma permite buscar directamente por vector o por texto
    resultados = collection.query(
        query_embeddings=[vector_lista],
        n_results=k
    )
    
    # Formatear la salida 
    formatted_results = []
    for i in range(len(resultados['ids'][0])):
        formatted_results.append((
            resultados['ids'][0][i],
            resultados['distances'][0][i], # Retorna distancia (1 - coseno)
            resultados['documents'][0][i],
            resultados['metadatas'][0][i]
        ))
        
    return formatted_results

# 5. Ejecutar consulta con k=5
resultados_chroma = chroma_search(query_vec, k=5)

# Imprimir resultados
print(f"\nResultados de la búsqueda en Chroma para 'Battery measuring':\n" + "="*60)
for i, (doc_id, distance, text, metadata) in enumerate(resultados_chroma):
    # Para similitud coseno, la distancia en Chroma es (1 - similitud), por lo que valores más cercanos a 0 son mejores.
    print(f"[{i+1}] ID: {doc_id} | Distancia Coseno: {distance:.4f}")
    print(f"    Doc ID: {metadata['doc_id']} | Chunk ID: {metadata['chunk_id']}")
    print(f"    Texto: {text[:200]}...\n")

Preparando listas para ChromaDB...
Insertando 79104 fragmentos en lotes de 5000...
Inserción en ChromaDB completada.

Resultados de la búsqueda en Chroma para 'Battery measuring':
[1] ID: 10176 | Distancia Coseno: 0.1297
    Doc ID: 1391 | Chunk ID: 0
    Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing the charge actually present in the cells and/or it...

[2] ID: 1 | Distancia Coseno: 0.1382
    Doc ID: 1 | Chunk ID: 0
    Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It...

[3] ID: 10177 | Distancia Coseno: 0.1599
    Doc ID: 1391 | Chunk ID: 1
    Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is based on the empiric

**¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?**

Fue considerablemente más rápido y directo. ChromaDB abstrae casi toda la complejidad del diseño de la base de datos. No obliga a instanciar clases de esquemas (como Weaviate), ni a definir tipos de datos rígidos para cada columna (como Milvus). Al funcionar de manera nativa con listas y diccionarios de Python, permite pasar de un DataFrame de Pandas a un índice vectorial funcional en un par de líneas de código.

**¿Qué limitaciones ves para un sistema en producción?**

Chroma es excelente para prototipos, RAGs locales o proyectos académicos, pero presenta varios cuellos de botella en entornos empresariales pesados:

Escalabilidad y Distribución: A diferencia de Milvus (que puede distribuir la carga en clusters masivos), ChromaDB tradicionalmente opera en un entorno single-node (un solo servidor).

Controles de Acceso (RBAC): Carece de sistemas robustos y granulares de roles, usuarios y permisos para segmentar quién puede acceder o modificar qué colecciones, algo vital en arquitecturas Zero Trust empresariales.

Consumo de Recursos: En colecciones de cientos de millones de vectores, la gestión de memoria RAM y almacenamiento no está tan optimizada como en bases de datos escritas puramente en C/Rust diseñadas específicamente para infraestructuras en la nube.

## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?


In [ ]:
import psycopg2
from psycopg2.extras import execute_batch
from pgvector.psycopg2 import register_vector
import numpy as np

# 1. Conectar a PostgreSQL
conn = psycopg2.connect(
    dbname="tu_base_de_datos", 
    user="postgres", 
    password="password", 
    host="localhost", 
    port="5432"
)
conn.autocommit = True
cur = conn.cursor()

# 2. Habilitar pgvector y registrar el tipo de dato en la conexión de Python
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
register_vector(conn)

# 3. Crear la tabla 'documents' con la columna de embedding
tabla_name = "wiki_documents"
cur.execute(f"DROP TABLE IF EXISTS {tabla_name};")

cur.execute(f"""
    CREATE TABLE {tabla_name} (
        id SERIAL PRIMARY KEY,
        doc_id INT,
        chunk_id INT,
        text TEXT,
        embedding vector(768) -- Dimensión exacta del modelo e5-base-v2
    );
""")

print("Tabla creada exitosamente. Preparando inserción...")

# 4. Insertar todos los documentos y embeddings
# Preparamos los datos en una lista de tuplas para execute_batch
insert_data = []
for idx, row in chunks_df.iterrows():
    insert_data.append((
        int(row["doc_id"]),
        int(row["chunk_id"]),
        row["text"],
        embeddings[idx] # psycopg2 con pgvector mapea automáticamente los arrays de numpy
    ))

# Inserción rápida en lotes
insert_query = f"INSERT INTO {tabla_name} (doc_id, chunk_id, text, embedding) VALUES (%s, %s, %s, %s)"
execute_batch(cur, insert_query, insert_data, page_size=1000)

print(f"Inserción de {len(insert_data)} registros completada.")

# 5. Función pgvector_search
def pgvector_search(query_embedding, k=5):
    # Asegurar formato unidimensional
    vector_lista = query_embedding[0] if len(query_embedding.shape) > 1 else query_embedding
    
    # El operador <=> calcula la distancia Coseno en pgvector
    search_query = f"""
        SELECT 
            id, 
            (embedding <=> %s::vector) AS distance, 
            text, 
            doc_id, 
            chunk_id
        FROM {tabla_name}
        ORDER BY distance
        LIMIT %s;
    """
    
    cur.execute(search_query, (vector_lista, k))
    resultados = cur.fetchall()
    
    # Formatear salida: id, score/distancia, text, metadata
    formatted_results = []
    for fila in resultados:
        rec_id, distance, text, doc_id, chunk_id = fila
        formatted_results.append((
            rec_id, 
            distance, 
            text, 
            {"doc_id": doc_id, "chunk_id": chunk_id}
        ))
        
    return formatted_results

# 6. Ejecutar consulta con k=5
resultados_sql = pgvector_search(query_vec, k=5)

print(f"\nResultados de la búsqueda en PostgreSQL/pgvector para 'Battery measuring':\n" + "="*60)
for i, (rec_id, score, text, metadata) in enumerate(resultados_sql):
    print(f"[{i+1}] ID en BD: {rec_id} | Distancia Coseno: {score:.4f}")
    print(f"    Doc ID: {metadata['doc_id']} | Chunk ID: {metadata['chunk_id']}")
    print(f"    Texto: {text[:200]}...\n")

cur.close()
conn.close()

Resultados de la búsqueda en PostgreSQL/pgvector para 'Battery measuring':
[1] ID en BD: 10177 | Distancia Coseno: 0.1297
    Doc ID: 1391 | Chunk ID: 0
    Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing the charge actually present in the cells and/or it...

[2] ID en BD: 2 | Distancia Coseno: 0.1382
    Doc ID: 1 | Chunk ID: 0
    Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It...

[3] ID en BD: 10178 | Distancia Coseno: 0.1599
    Doc ID: 1391 | Chunk ID: 1
    Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is based on the empirical fact that after having applie...

[4] ID en BD: 37407 | Distancia Coseno: 0.1609
   

**¿Qué tan “explicable” te parece esta aproximación vs las otras?**

Es la aproximación más explicable y transparente de todas. En motores especializados (como FAISS o Milvus), la recuperación ocurre en una "caja negra" optimizada a nivel de C++. Con PostgreSQL, la búsqueda es una sentencia declarativa explícita y auditable: le indicas a la base de datos exactamente la función matemática a utilizar mediante el operador lógico correspondiente (el operador <=> representa conceptualmente $argmin_{d \in D} \; \text{dist}(\vec{q}, \vec{d})$ con métrica coseno) y le ordenas ordenar por ese resultado.

**¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?**

La ventaja absoluta es la consolidación del ecosistema transaccional. En lugar de mantener una base de datos relacional para usuarios/permisos y otra vectorial para textos (y tener que sincronizar IDs entre ambas), en SQL puedes hacer todo en la misma consulta. Puedes ejecutar un JOIN para buscar similitud semántica solo en documentos cuyo autor tenga un rol específico, aplicar filtros de fecha precisos (WHERE date > '2023-01-01') y agrupar resultados (GROUP BY) sin preocuparte por el post-filtering o pre-filtering complejo que requieren los motores dedicados.

**¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?**

La principal limitación es el consumo de RAM y la eficiencia computacional en infraestructuras distribuidas. Bases como Milvus o Qdrant nacieron para el escalado horizontal nativo (sharding) y la gestión de billones de vectores. PostgreSQL, aunque pgvector soporta índices HNSW y optimizaciones IVFFlat, no está diseñado arquitectónicamente para mantener índices vectoriales masivos en memoria con la misma eficiencia, lo que puede provocar bloqueos (locks) o degradación del rendimiento general si el volumen transaccional de texto y vectores compite agresivamente con otras operaciones de la misma base de datos.